# 12. Team Operation (running & resetting)

Now that we can build a team, let's learn how to **operate** it day to day:

- **Run** a team and read the result.
- **Watch** it live with `Console` (like notebook 05, but for teams).
- **Reset** a team so it forgets the last job and starts fresh.

## Real-life analogy

A team is like a **meeting room**:

- **Run** = hold the meeting and get the outcome.
- **Watch live** = sit in and listen to the discussion as it happens.
- **Reset** = clear the whiteboard so the next meeting starts clean.

## The 3 operations

| Operation | Code | Meaning |
|-----------|------|---------|
| Run | `await team.run(task=...)` | Do the job, get the final result |
| Watch live | `await Console(team.run_stream(task=...))` | See every turn as it happens |
| Reset | `await team.reset()` | Clear memory before a new, unrelated task |

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

writer = AssistantAgent(name="writer", model_client=model_client,
                        system_message="Write one short slogan. Improve it if asked.")
reviewer = AssistantAgent(name="reviewer", model_client=model_client,
                          system_message="If the slogan is catchy, reply APPROVE. Else suggest one change.")

team = RoundRobinGroupChat([writer, reviewer],
                           termination_condition=TextMentionTermination("APPROVE"))

# Watch the team work LIVE with Console
await Console(team.run_stream(task="Create a slogan for a coffee shop."))

## Reset before a new, unrelated task

If you want the team to start a **fresh** job (not continue the old chat), call `reset()` first.

In [ ]:
# Clear the team's memory of the coffee slogan
await team.reset()

# Now run a brand-new, unrelated task
result = await team.run(task="Create a slogan for a book store.")
print(result.messages[-1].content)

## Key points to remember

- **Run** a team with `await team.run(task=...)` — same as a single agent.
- **Watch live** with `await Console(team.run_stream(task=...))`.
- **Reset** with `await team.reset()` to clear the chat before a new, unrelated task.
- Without reset, the team **continues** the previous conversation.